In [ ]:
import boto3

# Veficicación del usuario con el que se ejecuta el notebook

In [ ]:
sts = boto3.client("sts")
identity = sts.get_caller_identity()

In [ ]:
print(identity)

# Creación del ciente redshift

In [ ]:
redshift = boto3.client('redshift', region_name='us-east-1')

## Ejecutar este codigo solo 1 vez y que crea el servicio de redshift

In [ ]:
try:
    response = redshift.create_cluster(
        ClusterIdentifier="bsg-cluster-3",
        NodeType="ra3.xlplus",
        ClusterType="single-node",
        MasterUsername="awsuser",
        MasterUserPassword="123.Abc*.*",
        DBName="dev",
        PubliclyAccessible=True
    )
    print("Cluster solicitado correctamente")
    print(response)

except Exception as e:
    print("Código:", e.response["Error"]["Code"])
    print("Mensaje:", e.response["Error"]["Message"])

In [ ]:
print(response)

## Pruebas de conexion al clustes de redshifts

In [ ]:
import socket

socket.gethostbyname(
    "bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com"
)

In [ ]:
socket.create_connection(
    ("bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com", 5439),
    timeout=5
)

## Conectar al cluster Redshift

In [ ]:
import redshift_connector

In [ ]:
conn = redshift_connector.connect(
    host='bsg-cluster-3.cdvovwnpoijz.us-east-1.redshift.amazonaws.com',
    port=5439,
    database='dev',
    user='awsuser',
    password='123.Abc*.*'
)

In [ ]:
print(conn)

In [ ]:
cursor = conn.cursor()

### Crear una tabla de ejemplo

In [ ]:

cursor.execute("""
CREATE TABLE ventas (
    id INT,
    fecha DATE,
    monto DECIMAL(10,2),
    categoria VARCHAR(50)
)
""")

conn.commit()

### Agregar un Rol existente al redshift

In [ ]:
response = redshift.modify_cluster_iam_roles(
    ClusterIdentifier="bsg-cluster-3",
    AddIamRoles=[
        "arn:aws:iam::102317010484:role/LabRole"
    ]
)

### Copiar datos desde un CSV en S3 a la tabla (asumiendo el CSV no tiene cabecera y campos separados por coma)

In [ ]:
try:
    cursor.execute("""
    COPY ventas
    FROM 's3://mi-bucket-datos-bsg/ventas2025.csv'
    IAM_ROLE 'arn:aws:iam::102317010484:role/myRedshiftRole'
    FORMAT AS CSV
    DELIMITER ','
    IGNOREHEADER 1;
    """)
    conn.commit()
    print("Carga exitosa")
except Exception as e:
    conn.rollback()
    print("Error:", e)

### Consultas SQL básicas: Ahora podemos ejecutar consultas SQL con el cursor. Por ejemplo, contar filas o hacer agregaciones:

In [ ]:
cursor.execute("SELECT categoria, SUM(monto) as total FROM ventas GROUP BY 1;")
result = cursor.fetchall()
for row in result:
    print(row)